constructed a comprehensive, multi-level feature engineering pipeline for detecting deceptive reviews.

The objective was to build independent, leakage-free signals that capture:

Linguistic patterns

User behavior

Product context

Temporal activity

In [ ]:
import pandas as pd
import numpy as np
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import joblib

In [9]:
df = pd.read_csv("../data/processed/trust_scored_dataset.csv")

print("Shape:", df.shape)
df.head()

Shape: (882403, 22)


,user_id,product_id,rating,review_text,summary,verified,review_timestamp,clean_review_text,review_date,review_length,...,daily_count,rule_high_frequency,product_mean_rating,rating_deviation,rule_rating_deviation,rule_duplicate,fake_score,fake_label,label_confidence,trust_score
0,A1D4G1SNUZWQOT,7106116521,5,exactly what i needed.,perfect replacements!!,True,1413763200,exactly what i needed.,2014-10-20,4,...,1,0,3.823529,1.176471,0,1,3,1,high_fake,0.500000
1,A3DDWDH9PX2YX2,7106116521,2,"i agree with the other review, the opening is ...","I agree with the other review, the opening is ...",True,1411862400,"i agree with the other review, the opening is ...",2014-09-28,46,...,1,0,3.823529,1.823529,0,0,0,0,high_real,1.000000
2,A2MWC41EW7XL15,7106116521,4,love these... i am going to order another pack...,My New 'Friends' !!,False,1408924800,love these... i am going to order another pack...,2014-08-25,45,...,1,0,3.823529,0.176471,0,0,0,0,high_real,1.000000
3,A2UH2QQ275NV45,7106116521,2,too tiny an opening,Two Stars,True,1408838400,too tiny an opening,2014-08-24,4,...,1,0,3.823529,1.823529,0,0,0,0,high_real,1.000000
4,A89F3LQADZBS5,7106116521,3,okay,Three Stars,False,1406419200,okay,2014-07-27,1,...,1,0,3.823529,0.823529,0,1,2,1,uncertain,0.666667


### Basic Checks

In [10]:
print("Columns:\n", df.columns)
print("\nMissing values:\n", df.isnull().sum())

Columns:
 Index(['user_id', 'product_id', 'rating', 'review_text', 'summary', 'verified',
       'review_timestamp', 'clean_review_text', 'review_date', 'review_length',
       'rule_short_extreme', 'review_day', 'daily_count',
       'rule_high_frequency', 'product_mean_rating', 'rating_deviation',
       'rule_rating_deviation', 'rule_duplicate', 'fake_score', 'fake_label',
       'label_confidence', 'trust_score'],
      dtype='object')

Missing values:
 user_id                    0
product_id                 0
rating                     0
review_text                7
summary                  513
verified                   0
review_timestamp           0
clean_review_text          7
review_date                0
review_length              0
rule_short_extreme         0
review_day                 0
daily_count                0
rule_high_frequency        0
product_mean_rating        0
rating_deviation           0
rule_rating_deviation      0
rule_duplicate             0
fake_score      

### Standardize Column Names

In [11]:
df.rename(columns={
    'clean_review_text': 'clean_review'
}, inplace=True)

### Handle Missing Values

In [12]:
df['clean_review'] = df['clean_review'].fillna("")
df['verified'] = df['verified'].fillna(0)
df['review_length'] = df['review_length'].fillna(0)
df['rating_deviation'] = df['rating_deviation'].fillna(0)

df['review_date'] = pd.to_datetime(df['review_date'])

## NLP FEATURES

### Sentiment Polarity

In [13]:
analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    return analyzer.polarity_scores(text)['compound']

df['sentiment_score'] = df['clean_review'].apply(get_sentiment)

### Sentiment Extremeness

In [14]:
df['sentiment_extreme'] = abs(df['sentiment_score'])

### Repetition Ratio

In [15]:
def repetition_ratio(text):
    words = text.split()
    if len(words) == 0:
        return 0
    return 1 - (len(set(words)) / len(words))

df['repetition_ratio'] = df['clean_review'].apply(repetition_ratio)

### TF-IDF (Top 5000, 1-2 grams)

In [16]:
tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    max_features=5000,
    stop_words='english'
)

X_tfidf = tfidf.fit_transform(df['clean_review'])

print("TF-IDF shape:", X_tfidf.shape)

TF-IDF shape: (882403, 5000)


## USER BEHAVIORAL FEATURES

### User Review Count

In [17]:
df['user_review_count'] = df.groupby('user_id')['user_id'].transform('count')

### User Rating Variance

In [18]:
df['user_rating_variance'] = df.groupby('user_id')['rating'].transform('var').fillna(0)

### User Avg Rating Deviation

In [19]:
df['user_avg_rating_deviation'] = df.groupby('user_id')['rating_deviation'].transform('mean')

### User Review Frequency

In [20]:
df['user_first_review'] = df.groupby('user_id')['review_date'].transform('min')

df['days_active'] = (df['review_date'] - df['user_first_review']).dt.days + 1

df['user_review_frequency'] = df['user_review_count'] / df['days_active']

## PRODUCT CONTEXT FEATURES

### Product Review Count

In [21]:
df['product_review_count'] = df.groupby('product_id')['product_id'].transform('count')

### Product Rating Variance

In [22]:
df['product_rating_variance'] = df.groupby('product_id')['rating'].transform('var').fillna(0)

## TEMPORAL FEATURES

### Days Since First Product Review

In [23]:
df['product_first_review'] = df.groupby('product_id')['review_date'].transform('min')

df['days_since_first_review'] = (
    df['review_date'] - df['product_first_review']
).dt.days

### Review Density

In [24]:
df['review_density'] = df['product_review_count'] / (df['days_since_first_review'] + 1)

### Burst Indicator

In [32]:
threshold = df['daily_count'].mean() + df['daily_count'].std()
df['burst_indicator'] = (df['daily_count'] > threshold).astype(int)

## Select Structured Features

In [33]:
structured_features = df[
    [
        # NLP structured
        'review_length',
        'sentiment_score',
        'sentiment_extreme',
        'repetition_ratio',
        'rating_deviation',
        'verified',

        # User
        'user_review_count',
        'user_rating_variance',
        'user_avg_rating_deviation',
        'user_review_frequency',

        # Product
        'product_review_count',
        'product_rating_variance',

        # Temporal
        'days_since_first_review',
        'review_density',
        'burst_indicator'
    ]
]

structured_features.head()

,review_length,sentiment_score,sentiment_extreme,repetition_ratio,rating_deviation,verified,user_review_count,user_rating_variance,user_avg_rating_deviation,user_review_frequency,product_review_count,product_rating_variance,days_since_first_review,review_density,burst_indicator
0,4,0.0000,0.0000,0.000000,1.176471,True,1,0.0,1.176471,1.0,17,2.154412,142,0.118881,0
1,46,0.1901,0.1901,0.152174,1.823529,True,2,4.5,0.911765,2.0,17,2.154412,120,0.140496,0
2,45,0.8074,0.8074,0.088889,0.176471,False,1,0.0,0.176471,1.0,17,2.154412,86,0.195402,0
3,4,0.0000,0.0000,0.000000,1.823529,True,1,0.0,1.823529,1.0,17,2.154412,85,0.197674,0
4,1,0.2263,0.2263,0.000000,0.823529,False,1,0.0,0.823529,1.0,17,2.154412,57,0.293103,0


### Scale Structured Features

In [27]:
scaler = StandardScaler()
X_structured = scaler.fit_transform(structured_features)

print("Structured shape:", X_structured.shape)

Structured shape: (882403, 15)


### Combine TF-IDF + Structured

In [28]:
X = hstack([X_tfidf, X_structured])
y = df['trust_score']

print("Final Feature Shape:", X.shape)

Final Feature Shape: (882403, 5015)


### Train Test Split

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (705922, 5015)
Test shape: (176481, 5015)


### Save Artifacts

In [30]:
os.makedirs("../models", exist_ok=True)

joblib.dump(tfidf, "../models/tfidf_vectorizer.pkl")
joblib.dump(scaler, "../models/feature_scaler.pkl")

print("Artifacts saved.")

Artifacts saved.
